# Notebook 16 — Pretraining a Random-Initialized Hugging Face Model

    ## Learning objectives

    - Instantiate a Transformers causal LM from configuration rather than downloaded weights
- Run a one-epoch Hugging Face-compatible pretraining loop and save_pretrained artifact
- Separate architecture, tokenizer, weights, training recipe, and downstream usability

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = ['transformers>=4.51,<5', 'datasets>=3.5,<6']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from the Colab Secrets UI without displaying it. Create a
# secret named exactly HF_TOKEN and enable notebook access with its toggle.
token = os.getenv("HF_TOKEN")
token_error = None
if IN_COLAB and not token:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        token_error = type(exc).__name__
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub environment variable.
if token:
    os.environ["HF_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if True and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("HF_TOKEN is unavailable. In Colab, open the key icon (Secrets), add HF_TOKEN, ")
    print("enable its Notebook access toggle, and rerun this cell. Public models still work.")
    if token_error:
        print("Colab secret lookup status:", token_error)


## 16.1 `from_config` means architecture without learned knowledge

`from_pretrained` loads a configuration plus learned parameters. `from_config` constructs the
same kind of module with freshly initialized parameters. We deliberately combine a mature GPT-2
tokenizer with a very small GPT-2-shaped decoder. Reusing a tokenizer is convenient but does not
transfer the source model's language knowledge: the embedding rows and transformer weights are
random. The experiment therefore remains pretraining from scratch at the model-weight level.

Architecture size, tokenizer choice, corpus, context length, optimizer, and compute budget are
independent design axes. A production pretraining run needs held-out and contamination-controlled
evaluation, licensed and documented data, distributed checkpointing, resumability, and many more
tokens than this pedagogical run.


In [ ]:
import math, torch
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
TOKENIZER_ID = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
tokenizer.pad_token = tokenizer.eos_token
config = AutoConfig.for_model("gpt2", vocab_size=len(tokenizer), n_positions=96, n_ctx=96,
                              n_embd=128, n_layer=3, n_head=4,
                              bos_token_id=tokenizer.bos_token_id, eos_token_id=tokenizer.eos_token_id)
model = AutoModelForCausalLM.from_config(config).to(device)  # random weights
print(type(model).__name__, f"parameters={model.num_parameters():,}", "device=", device)


In [ ]:
corpus = [
    "Language models learn distributions over token sequences.",
    "A causal mask prevents attention from reading future positions.",
    "The optimizer changes parameters using gradients of prediction loss.",
    "Held-out loss measures prediction on text excluded from optimization.",
    "Checkpoints pair weights with configuration and tokenizer artifacts.",
    "Tiny demonstrations teach mechanics rather than general language ability.",
]
def make_blocks(texts, repeats, block_size=64):
    stream = []
    for _ in range(repeats):
        for text in texts: stream += tokenizer(text + tokenizer.eos_token, add_special_tokens=False).input_ids
    usable = len(stream) // block_size * block_size
    return [torch.tensor(stream[i:i+block_size]) for i in range(0, usable, block_size)]

class Blocks(Dataset):
    def __init__(self, values): self.values = values
    def __len__(self): return len(self.values)
    def __getitem__(self, i): return {"input_ids": self.values[i]}

train_blocks = make_blocks(corpus[:5], repeats=24)
valid_blocks = make_blocks(corpus[5:], repeats=8)
def causal_collator(records):
    # Blocks are equal length: stack directly and retain real EOS labels. A generic
    # collator may mask every EOS when EOS is also configured as the padding token.
    ids = torch.stack([record["input_ids"] for record in records])
    return {"input_ids": ids, "attention_mask": torch.ones_like(ids), "labels": ids.clone()}
train_loader = DataLoader(Blocks(train_blocks), batch_size=8, shuffle=True, collate_fn=causal_collator)
valid_loader = DataLoader(Blocks(valid_blocks), batch_size=8, collate_fn=causal_collator)
print("train blocks:", len(train_blocks), "valid blocks:", len(valid_blocks))


In [ ]:
@torch.no_grad()
def complete(prompt, max_new_tokens=30):
    model.eval(); batch = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=True,
                            temperature=0.8, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0], skip_special_tokens=True)
print("BEFORE:", repr(complete("A language model")))


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
model.train(); losses = []
for batch in train_loader:                          # exactly one epoch
    batch = {key: value.to(device) for key, value in batch.items()}
    optimizer.zero_grad(set_to_none=True)
    output = model(**batch)
    output.loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step(); losses.append(output.loss.item())
print(f"one-epoch mean train loss: {sum(losses)/len(losses):.3f}")


In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval(); total, batches = 0.0, 0
    for batch in loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        total += model(**batch).loss.item(); batches += 1
    loss = total / batches
    return {"loss": loss, "perplexity": math.exp(min(loss, 20))}
print("validation:", evaluate(valid_loader))
print("AFTER:", repr(complete("A language model")))


In [ ]:
output_dir = Path("artifacts/hf_random_pretraining")
model.save_pretrained(output_dir, safe_serialization=True)
tokenizer.save_pretrained(output_dir)
reloaded = AutoModelForCausalLM.from_pretrained(output_dir).to(device)
probe = tokenizer("A causal mask", return_tensors="pt").to(device)
model.eval(); reloaded.eval()
with torch.no_grad():
    delta = (model(**probe).logits - reloaded(**probe).logits).abs().max().item()
print("saved files:", sorted(p.name for p in output_dir.iterdir()), "max logit delta:", delta)


## 16.2 Interpreting the result

Falling training loss proves the pipeline can fit this stream. It does not establish facts,
instruction following, safety, or even robust English generation. The validation set here is
tiny and shares style with training; its perplexity is a smoke test, not a scientific result.
The saved directory is nevertheless a genuine Hugging Face model artifact: config, weights,
tokenizer, and generation metadata can be reloaded through standard APIs and used as the
starting point for continued pretraining or the post-training stages in the next notebook.


## 16.3 Hugging Face object boundaries

A tokenizer maps text and special-token conventions to IDs. A configuration specifies architecture
and dimensions. A model class implements computation and owns parameters. A checkpoint supplies
parameter values. `AutoModelForCausalLM.from_config(config)` selects the architecture and initializes
new weights; `from_pretrained(path_or_id)` resolves configuration and weights from a directory or
Hub repository. Confusing these calls can accidentally turn a from-scratch experiment into continued
pretraining—or discard expensive weights.

A language-model collator normally copies input IDs into labels and marks padding as ignored. Because
GPT-2 reuses EOS as PAD and our equal-length blocks need no padding, the explicit collator deliberately
retains genuine EOS labels. Blindly masking by token ID would erase every document boundary. Model
forward shifts logits and labels internally when `labels` are supplied. Inspect this behavior for any
custom architecture rather than shifting twice. `save_pretrained` writes a portable artifact, but optimizer,
scheduler, data position, RNG state, metrics, and code revision still need a training checkpoint or
experiment record if exact resumption matters.


In [ ]:
sample = next(iter(train_loader))
sample = {key: value.to(device) for key, value in sample.items()}
with torch.no_grad(): out = model(**sample)
print("logits:", tuple(out.logits.shape), "labels:", tuple(sample["labels"].shape))
print("ignored label positions:", int((sample["labels"] == -100).sum()))
print("config model_type:", model.config.model_type,
      "tied embeddings:", model.config.tie_word_embeddings)


## 16.4 From a local artifact to a governed model release

A useful repository should include a model card, exact base/tokenizer references, licenses, dataset
lineage, intended and excluded uses, training hyperparameters, evaluation tables, limitations, and
example loading code. Pin revisions when reproducing a run. Safe tensor serialization avoids pickle
execution for weight files, but consumers must still review custom code, dependencies, and model
provenance. A Hub token is needed only for gated/private resources or upload; it must come from
Colab Secrets or environment variables and must never be embedded in the notebook.

Continued pretraining starts from learned weights and exposes them to more raw domain text. It can
improve domain likelihood yet cause catastrophic forgetting or alter safety behavior. Instruction
tuning instead trains desired prompt-response behavior, usually masking prompt labels. The next
lesson maps those branches before later notebooks implement LoRA SFT and DPO.

Initialization is part of the reproducibility contract. A fixed PyTorch seed makes this lesson easier
to debug, but accelerator kernels and software versions can still alter exact results. Store package
versions, configuration JSON, corpus fingerprint, seed, precision, and hardware alongside metrics.
When comparing architectures, match useful token or compute budgets and repeat multiple seeds; one
lucky small run is weak evidence.

Generation before and after training is intentionally sampled. For a strict regression test, compare
teacher-forced loss or fixed logits on frozen inputs. For a model card, show multiple representative
prompts, disclose decoding parameters, and include failures. Never present a memorized training phrase
as evidence of broad instruction following. This base artifact has not learned a chat template,
alignment policy, factual coverage, calibrated uncertainty, or reliable stopping behavior.


## 16.5 Configuration-to-artifact contract

Random initialization should begin from an explicit Transformers configuration whose vocabulary size, special-token IDs, context limit, width, heads, layers, and tying policy match the tokenizer and budget. Count parameters immediately and save the untrained configuration as part of the run manifest. After training, save model and tokenizer together, reload with `from_pretrained`, and compare fixed logits. Loading a familiar architecture class does not import pretrained knowledge when constructed from configuration.


In [ ]:
config_contract={"vocab_size":512,"bos_token_id":1,"eos_token_id":2,"n_positions":128,"n_embd":128,"n_layer":4,"n_head":4,"tie_word_embeddings":True}
assert config_contract["n_embd"]%config_contract["n_head"]==0; print(config_contract)


## 16.6 Trainer accounting and callbacks

High-level trainers still require verification of label shifting, ignored positions, effective tokens per update, scheduler horizon, evaluation cadence, checkpoint retention, and resume semantics. `gradient_accumulation_steps` changes optimizer-update frequency; warmup and logging strategies may count updates rather than microbatches. Add callbacks or logs for valid-token counts, throughput, gradient norms, and resolved revisions. Inspect one batch and one manual loss before `train()`, then compare trainer-reported evaluation loss with a direct token-weighted pass.


In [ ]:
trainer_plan={"microbatch":4,"sequence":128,"accumulation":8,"devices":1,"updates":1000}
trainer_plan["tokens_per_update"]=trainer_plan["microbatch"]*trainer_plan["sequence"]*trainer_plan["accumulation"]*trainer_plan["devices"]
trainer_plan["scheduled_tokens"]=trainer_plan["tokens_per_update"]*trainer_plan["updates"]; print(trainer_plan)


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [Transformers model creation](https://huggingface.co/docs/transformers/create_a_model)
- [Transformers Trainer](https://huggingface.co/docs/transformers/trainer)


## Exercises

    1. Change only model width and compare parameters, step time, and held-out loss.
2. Train a tokenizer on the same corpus and explain why the result is not a fair quality comparison.
3. Resume from the saved artifact for a second epoch and preserve optimizer state separately.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
